# 17 · Memory needs correction and forgetting

## The engineering question
Can retrieval honor a correction, expiration, and deletion even when obsolete text is semantically close?

**Learning outcomes:** distinguish episodic records from current facts; retain correction provenance; filter by validity time and tenant; propagate deletion to the index; reject replay of a tombstoned revision.

The fictional travel preference changes from window to aisle. We treat the correction as a new revision of the same source, keep the event history in the ingestion manifest, and remove obsolete points. A temporary appointment record expires at a synthetic timestamp. Semantic similarity provides ranking only; currentness and permission are explicit constraints. This is a memory lifecycle experiment, not a claim that a model has human memory.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("17", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from uuid import uuid4
from vector_masterclass.schema import Document
from vector_masterclass.store import VectorIndex
from vector_masterclass.ingestion import Ingestor

index = VectorIndex("vmc_17_" + uuid4().hex[:8])
index.create()
ingest = Ingestor(index, lesson.path / "memory-events.sqlite")
old = Document(
    id="seat-preference",
    revision="1",
    title="Seat preference",
    text="The red traveler prefers a window seat.",
    tenant="red",
    published="2026-01-01",
)
corrected = old.model_copy(
    update={
        "revision": "2",
        "text": "The red traveler corrected the preference: choose an aisle seat. The earlier window preference is superseded.",
        "published": "2026-02-01",
    }
)
appointment = Document(
    id="appointment",
    revision="1",
    title="Temporary appointment",
    text="The red traveler has a temporary appointment at 09:00.",
    tenant="red",
)
other = Document(
    id="blue-preference",
    revision="1",
    title="Blue traveler",
    text="The blue traveler prefers a window seat.",
    tenant="blue",
)
rows = []
current = {old.id: old, appointment.id: appointment}
expires = {old.id: float("inf"), appointment.id: 120}


def retrieve(now):
    allowed = [doc.key for id_, doc in current.items() if expires[id_] > now]
    return index.search(
        "What seat does the traveler prefer and what appointment is planned?",
        tenant="red",
        document_keys=allowed,
        k=10,
    )


try:
    ingest.ingest([old, appointment, other])
    before = retrieve(100)
    rows.append(
        {
            "stage": "initial",
            "retrieved_sources": [h.chunk.document_key for h in before],
            "visible_points": len(index.all_points()),
        }
    )
    ingest.ingest([corrected])
    current[old.id] = corrected
    after = retrieve(100)
    rows.append(
        {
            "stage": "after correction",
            "retrieved_sources": [h.chunk.document_key for h in after],
            "visible_points": len(index.all_points()),
        }
    )
    expired = retrieve(120)
    rows.append(
        {
            "stage": "after expiry filter",
            "retrieved_sources": [h.chunk.document_key for h in expired],
            "visible_points": len(index.all_points()),
        }
    )
    ingest.delete(corrected.id, "red")
    del current[corrected.id]
    deleted = retrieve(120)
    rows.append(
        {
            "stage": "after explicit deletion",
            "retrieved_sources": [h.chunk.document_key for h in deleted],
            "visible_points": len(index.all_points()),
        }
    )
    rejected = False
    try:
        ingest.ingest([corrected])
    except ValueError:
        rejected = True
    events = lesson.table(pd.DataFrame(ingest.events()).drop(columns=["source_json"]), "events")
    consistent = ingest.reconcile()["consistent"]
finally:
    ingest.close()
    index.drop()
    index.close()
results = lesson.table(pd.DataFrame(rows))
display(results)
display(events[["sequence", "source_id", "operation", "state"]])
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(results.stage, [len(ids) for ids in results.retrieved_sources])
ax.set(
    ylabel="Eligible retrieved records",
    title="Validity and deletion constrain memory before ranking",
)
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Expiration hides a record from retrieval while leaving stored points until cleanup; deletion removes indexed points and leaves a tombstone/event for reconciliation. The correction retains history for audit while preferring the current record. Audit retention and erasure requirements must be defined separately, including source logs, backups, caches, and derived indexes.

**Exercise:** move the clock just before and exactly to expiration. Then attempt to reingest the deleted revision. Explain why an ordered event log or source-version authority is needed to reject older events in a multi-writer system. The single-writer teaching pipeline does not infer ordering from arbitrary revision strings.

Memory ranking can combine similarity with recency and importance, but those heuristics must operate only after tenant, validity, correction, and deletion constraints. Do not preserve an obsolete fact merely because it has the highest score.

In [ ]:
manifest = lesson.finish(
    {
        "old_revision_removed": all(h.chunk.document_key != "seat-preference@1" for h in after),
        "correction_present": any(h.chunk.document_key == "seat-preference@2" for h in after),
        "expiry_enforced": all(h.chunk.document_key != "appointment@1" for h in expired),
        "deleted_memory_absent": len(deleted) == 0,
        "tenant_boundary": all(h.chunk.tenant == "red" for h in before + after + expired),
        "tombstone_replay_rejected": rejected,
        "manifest_converged": consistent,
    },
    observations={
        "clock": "synthetic seconds 100..120",
        "expiry_is_filtering_not_physical_erasure": True,
        "history_retention": "SQLite teaching event log; explicit production retention decision required",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.